<a href="https://colab.research.google.com/github/Dani2003/paper-implementations/blob/main/Implementation_of_DISCRETE_DIFFUSION_MODELS_WITH_MLLMS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# [CELL 1 - Environment Setup]
!pip install -q torch torchvision transformers datasets matplotlib

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer
import numpy as np
import matplotlib.pyplot as plt
import math

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print("MeDiM (Medical Discrete Diffusion Model) environment initialized.")

Using device: cuda
MeDiM (Medical Discrete Diffusion Model) environment initialized.


In [2]:
# [CELL 2 - Discrete Tokenizers & Synthetic Dataset]
from transformers import GPT2TokenizerFast

# 1. Text Tokenizer
text_tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
text_tokenizer.pad_token = text_tokenizer.eos_token
TEXT_VOCAB_SIZE = len(text_tokenizer) # ~50,257

# 2. Visual Discrete Tokenizer Mock (Simulates VQ-VAE codebook)
VISUAL_VOCAB_SIZE = 1024  # Codebook size for images
NUM_VISUAL_TOKENS = 16    # 4x4 spatial patches per medical image

# Offsets to unify vocabulary into a single sequence: [Visual Tokens, Text Tokens, MASK]
VISUAL_OFFSET = 0
TEXT_OFFSET = VISUAL_VOCAB_SIZE
MASK_TOKEN_ID = VISUAL_VOCAB_SIZE + TEXT_VOCAB_SIZE  # Special Absorbing [MASK] token
TOTAL_VOCAB_SIZE = MASK_TOKEN_ID + 1

# Synthetic Dataset: Paired Medical Image Tokens + Clinical Report Text
synthetic_data = [
    {"image_tokens": [10, 45, 88, 102, 12, 44, 89, 101, 15, 46, 90, 103, 11, 47, 91, 104],
     "report": "Normal cardiothoracic ratio. Lungs are clear without focal consolidation."},
    {"image_tokens": [500, 512, 530, 600, 501, 513, 531, 601, 502, 514, 532, 602, 503, 515, 533, 603],
     "report": "Bilateral pleural effusion present. Moderate cardiomegaly noted."},
] * 50  # Expand dataset

def prepare_multimodal_sequence(item):
    # Map visual codebook indices
    vis_seq = [t + VISUAL_OFFSET for t in item["image_tokens"]]
    # Map text tokens
    text_ids = text_tokenizer.encode(item["report"], truncation=True, max_length=32)
    text_seq = [t + TEXT_OFFSET for t in text_ids]
    # Unified multimodal sequence [Vision; Text]
    full_seq = vis_seq + text_seq
    return torch.tensor(full_seq, dtype=torch.long)

dataset = [prepare_multimodal_sequence(item) for item in synthetic_data]
print(f"Dataset size: {len(dataset)} paired samples")
print(f"Total Unified Vocab Size: {TOTAL_VOCAB_SIZE}")
print(f"Sample sequence length: {len(dataset[0])} (Vision: {NUM_VISUAL_TOKENS}, Text: {len(dataset[0]) - NUM_VISUAL_TOKENS})")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Dataset size: 100 paired samples
Total Unified Vocab Size: 51282
Sample sequence length: 33 (Vision: 16, Text: 17)


In [3]:
# [CELL 3 - MeDiM Model Architecture]

class SinusoidalPosEmb(nn.Module):
    """Continuous Timestep Embedding (Sinusoidal)"""
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, x):
        device = x.device
        half_dim = self.dim // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=device) * -emb)
        emb = x[:, None] * emb[None, :]
        emb = torch.cat((emb.sin(), emb.cos()), dim=-1)
        return emb

class MeDiMBackbone(nn.Module):
    def __init__(self, vocab_size, d_model=256, nhead=8, num_layers=4):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Parameter(torch.zeros(1, 128, d_model)) # Max seq len 128

        # 1. Timestep Embedding Projection (Continuous t -> d_model)
        self.time_mlp = nn.Sequential(
            SinusoidalPosEmb(d_model),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model)
        )

        # 2. Bidirectional Transformer Encoder Blocks (No Causal Mask)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            activation="gelu",
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Output Logits Head
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, x, t):
        """
        x: [Batch, Seq_Len] - Corrupted discrete sequence
        t: [Batch] - Continuous timestep in [0, 1]
        """
        B, L = x.shape

        # Embed discrete tokens + positional embeddings
        h = self.token_emb(x) + self.pos_emb[:, :L, :]

        # Inject Continuous Timestep Embedding across sequence
        t_emb = self.time_mlp(t).unsqueeze(1) # [Batch, 1, d_model]
        h = h + t_emb

        # Bidirectional Context Flow (All tokens attend to all tokens)
        out = self.transformer(h) # Native TransformerEncoder is BIDIRECTIONAL

        logits = self.head(out) # [Batch, Seq_Len, Vocab_Size]
        return logits

# Model Verification
model = MeDiMBackbone(vocab_size=TOTAL_VOCAB_SIZE).to(device)
dummy_x = torch.randint(0, TOTAL_VOCAB_SIZE, (2, 30)).to(device)
dummy_t = torch.tensor([0.2, 0.85]).to(device)
dummy_out = model(dummy_x, dummy_t)

print(f"MeDiM Backbone initialized successfully.")
print(f"Output shape: {dummy_out.shape} (Batch, Seq_Len, Vocab_Size)")

MeDiM Backbone initialized successfully.
Output shape: torch.Size([2, 30, 51282]) (Batch, Seq_Len, Vocab_Size)


In [4]:
# [CELL 4 - Discrete Diffusion Scheduler & Loss]

class DiscreteAbsorbingDiffusion:
    def __init__(self, mask_token_id):
        self.mask_token_id = mask_token_id

    def q_sample(self, x_0, t):
        """
        Forward Noise Process: Corrupt x_0 to x_t by replacing tokens with [MASK]
        Probability of masking gamma(t) = t
        """
        B, L = x_0.shape
        # Mask probability increases linearly with t
        mask_prob = t.unsqueeze(-1).expand(B, L)

        # Random matrix to determine masked positions
        rand = torch.rand((B, L), device=x_0.device)
        mask_flags = rand < mask_prob

        x_t = x_0.clone()
        x_t[mask_flags] = self.mask_token_id
        return x_t, mask_flags

    def compute_loss(self, model, x_0):
        B, L = x_0.shape

        # Sample continuous timestep t ~ U(0.001, 0.999)
        t = torch.rand(B, device=x_0.device) * 0.998 + 0.001

        # Corrupt inputs
        x_t, mask_flags = self.q_sample(x_0, t)

        # Predict logits with MeDiM
        logits = model(x_t, t)

        # Cross Entropy Loss ONLY on the masked token positions
        loss = F.cross_entropy(
            logits[mask_flags],
            x_0[mask_flags]
        )
        return loss

diffusion_engine = DiscreteAbsorbingDiffusion(mask_token_id=MASK_TOKEN_ID)
test_loss = diffusion_engine.compute_loss(model, dummy_x)
print(f"Discrete Diffusion Forward Pass verified. Initial Loss: {test_loss.item():.4f}")

Discrete Diffusion Forward Pass verified. Initial Loss: 10.9984


In [ ]:
# [CELL 5 - MeDiM Training Execution]

from torch.utils.data import DataLoader

# Padding function for variable length sequences
def pad_collate(batch):
    max_len = max(len(s) for s in batch)
    padded = torch.full((len(batch), max_len), MASK_TOKEN_ID, dtype=torch.long)
    for i, s in enumerate(batch):
        padded[i, :len(s)] = s
    return padded

train_loader = DataLoader(dataset, batch_size=8, shuffle=True, collate_fn=pad_collate)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)

model.train()
epochs = 15

print("Starting MeDiM Training (Joint Vision-Language Discrete Diffusion)...")
print("=" * 65)

for epoch in range(1, epochs + 1):
    total_loss = 0.0
    for step, batch in enumerate(train_loader):
        batch = batch.to(device)
        optimizer.zero_grad()

        loss = diffusion_engine.compute_loss(model, batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    if epoch % 3 == 0 or epoch == 1:
        print(f"Epoch [{epoch:02d}/{epochs:02d}] | MeDiM Discrete Diffusion Loss: {avg_loss:.4f}")

print("=" * 65)
print("Training Complete. MeDiM is ready for generative inference.")

In [5]:
# [CELL 6 - MeDiM Multi-Modal Generative Inference]

@torch.no_grad()
def generate_medim(model, prompt_seq=None, target_len=30, num_steps=10, mode="joint"):
    """
    Iterative Reverse Discrete Diffusion Sampling
    """
    model.eval()

    # Initialize sequence with MASK tokens
    x = torch.full((1, target_len), MASK_TOKEN_ID, dtype=torch.long, device=device)

    # If conditional mode, fix prompt tokens (e.g. Image tokens provided)
    if prompt_seq is not None:
        p_len = len(prompt_seq)
        x[0, :p_len] = prompt_seq.to(device)

    timesteps = torch.linspace(1.0, 0.0, num_steps + 1, device=device)

    for i in range(num_steps):
        t_curr = timesteps[i].unsqueeze(0)

        # Predict logits for masked locations
        logits = model(x, t_curr)
        probs = F.softmax(logits, dim=-1)

        # Sample highest probability tokens
        pred_tokens = torch.argmax(probs, dim=-1)

        # Determine how many tokens to unmask at step i
        mask_indices = (x == MASK_TOKEN_ID)
        if mode == "report_from_image":
            # Do not touch image prompt tokens
            mask_indices[0, :len(prompt_seq)] = False

        # Gradually unmask tokens
        x[mask_indices] = pred_tokens[mask_indices]

    return x[0]

# --- Test 1: Image-to-Report Translation ---
sample_img_tokens = torch.tensor([10, 45, 88, 102, 12, 44, 89, 101, 15, 46, 90, 103, 11, 47, 91, 104])
gen_seq = generate_medim(model, prompt_seq=sample_img_tokens, target_len=30, mode="report_from_image")

# Extract generated text tokens
gen_text_ids = [t.item() - TEXT_OFFSET for t in gen_seq[NUM_VISUAL_TOKENS:] if TEXT_OFFSET <= t.item() < MASK_TOKEN_ID]
generated_report = text_tokenizer.decode(gen_text_ids)

print("\n" + "="*60)
print("MeDiM GENERATIVE BENCHMARK EVALUATION")
print("="*60)
print(f"Conditioning Visual Tokens (16 Patches) : {sample_img_tokens.tolist()[:4]}...")
print(f"Generated Medical Report               : {generated_report}")
print("="*60)


MeDiM GENERATIVE BENCHMARK EVALUATION
Conditioning Visual Tokens (16 Patches) : [10, 45, 88, 102]...
Generated Medical Report               :  fortunate fortunate fortunate fortunate fortunate fortunate fortunate fortunate fortunate fortunate fortunate fortunate fortunate fortunate
